# 09 — Runway Health Index fusion (PARTIAL — 3 of 4 signals)

Honest scope note up front: the IDF's Tier 4 fusion combines four signals
(survival probability, cash-flow projection, distress-language score,
anomaly indicator). The distress-language estimator (Tier 3's third
estimator) needs a founder/press text corpus that **does not exist in the
Crunchbase snapshot** — it has no text field at all. That's not a modeling
gap, it's a data-acquisition gap: scraping or licensing a founder-blog/press
corpus is a separate piece of work, not something this notebook can fake
with the data on hand. So this fusion runs on the three signals that ARE
real: survival probability, cash-flow projection, and the anomaly
indicator. Experiment 5 is reported as partial for this reason, and the
NLP signal slots in later without changing this structure.

Reads: `data/processed/survival_scored.csv`, `data/processed/cashflow_projections.csv`,
`data/processed/spend_anomalies.csv`
Writes: `reports/experiment5_fusion_partial.csv`

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

PROCESSED = "../data/processed"
REPORTS = "../reports"

survival = pd.read_csv(f"{PROCESSED}/survival_scored.csv")
cashflow = pd.read_csv(f"{PROCESSED}/cashflow_projections.csv")
anomalies = pd.read_csv(f"{PROCESSED}/spend_anomalies.csv")

anomaly_agg = anomalies.groupby("object_id")["is_anomaly"].mean().rename("anomaly_rate").reset_index()

merged = survival.merge(cashflow, left_on="id", right_on="object_id", how="inner") \
                  .merge(anomaly_agg, on="object_id", how="left")
merged["anomaly_rate"] = merged["anomaly_rate"].fillna(0)
print(f"[merge] {len(merged):,} companies have all three signals (survival + cash-flow + anomaly)")
print("[merge] this is a small subset of the full 98,280 -- only companies where the funding-round")
print("[merge] structure supported burn-trajectory synthesis in notebook 06 are included here.")

[merge] 1,892 companies have all three signals (survival + cash-flow + anomaly)
[merge] this is a small subset of the full 98,280 -- only companies where the funding-round
[merge] structure supported burn-trajectory synthesis in notebook 06 are included here.


In [2]:
# normalise the cash-flow signal onto a comparable 0-1 scale (lower months-to-zero = more urgent)
merged["cashflow_urgency"] = 1 / (1 + merged["months_to_zero_projected"].clip(lower=0) / 12)

signal_cols = ["p_exhaust_6m", "cashflow_urgency", "anomaly_rate"]
X = merged[signal_cols].fillna(0)
y = merged["event"]

# Partition B: held out specifically for fitting the fusion weights, distinct from
# the rows used to evaluate the survival model alone in notebook 03.
#
# A single train/test split is noisy at this sample size (a single unlucky split
# can make a real signal look inverted -- if you're seeing a single-signal AUC
# below 0.5, that is very likely this, not a broken model). Use 5-fold
# cross-validation instead, and report the mean +/- std across folds.
from sklearn.model_selection import StratifiedKFold, cross_val_score

if y.nunique() > 1 and y.value_counts().min() >= 5:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)
else:
    cv = 5

fusion_cv = LogisticRegression()
fused_scores = cross_val_score(fusion_cv, X, y, cv=cv, scoring="roc_auc")

single_scores = cross_val_score(LogisticRegression(), X[["p_exhaust_6m"]], y, cv=cv, scoring="roc_auc")

print(f"[Experiment 5, partial] fused (3-signal) AUC, 5-fold CV:      {fused_scores.mean():.3f} +/- {fused_scores.std():.3f}")
print(f"[Experiment 5, partial] best single signal AUC, 5-fold CV:    {single_scores.mean():.3f} +/- {single_scores.std():.3f}")
print(f"[Experiment 5, partial] per-fold fused:  {np.round(fused_scores, 3)}")
print(f"[Experiment 5, partial] per-fold single: {np.round(single_scores, 3)}")
print()
print("If the per-fold numbers above swing widely (e.g. one fold near 0.3, another near 0.8),")
print("that is itself the finding: this sample is too small/noisy for a confident verdict yet.")
print("Report the mean +/- std, not a single split's number.")

# fit the fusion model on everything for downstream RHI scoring
fusion = LogisticRegression()
fusion.fit(X, y)
fused_auc = fused_scores.mean()
best_single_auc = single_scores.mean()

if fused_auc > best_single_auc:
    print("\n[Experiment 5, partial] fusion beats the best single signal on average across folds.")
else:
    print("\n[Experiment 5, partial] fusion does NOT beat the best single signal on average -- report this")
    print("[Experiment 5, partial] as a genuine finding, per the IDF's own instruction, not a bug to hide.")

weights = dict(zip(signal_cols, fusion.coef_[0]))
print(f"\n[weights] (fit on full data for RHI scoring below) {weights}")

[Experiment 5, partial] fused (3-signal) AUC, 5-fold CV:      0.622 +/- 0.057
[Experiment 5, partial] best single signal AUC, 5-fold CV:    0.753 +/- 0.018
[Experiment 5, partial] per-fold fused:  [0.584 0.637 0.726 0.57  0.594]
[Experiment 5, partial] per-fold single: [0.74  0.778 0.761 0.727 0.759]

If the per-fold numbers above swing widely (e.g. one fold near 0.3, another near 0.8),
that is itself the finding: this sample is too small/noisy for a confident verdict yet.
Report the mean +/- std, not a single split's number.

[Experiment 5, partial] fusion does NOT beat the best single signal on average -- report this
[Experiment 5, partial] as a genuine finding, per the IDF's own instruction, not a bug to hide.

[weights] (fit on full data for RHI scoring below) {'p_exhaust_6m': np.float64(-0.2024148014851307), 'cashflow_urgency': np.float64(0.0764888948744246), 'anomaly_rate': np.float64(1.5039234561039483)}


In [3]:
# Runway Health Index: bounded 0-100, lower = more urgent (matches IDF's convention)
merged["rhi_raw"] = fusion.predict_proba(X)[:, 1]
merged["RHI"] = (100 * (1 - merged["rhi_raw"])).round(1)

summary = pd.DataFrame({
    "signals_used": ["survival probability (real)", "cash-flow projection (real, placeholder starting-cash)",
                      "anomaly rate (real)", "distress language (NOT AVAILABLE -- no text corpus)"],
    "status": ["included", "included", "included", "excluded -- Phase 2b data acquisition needed"],
})
summary.to_csv(f"{REPORTS}/experiment5_fusion_partial.csv", index=False)
print(summary.to_string(index=False))
print(f"\n[done] wrote {REPORTS}/experiment5_fusion_partial.csv")
print(f"[done] median RHI across {len(merged)} scored companies: {merged['RHI'].median():.1f}")

                                          signals_used                                       status
                           survival probability (real)                                     included
cash-flow projection (real, placeholder starting-cash)                                     included
                                   anomaly rate (real)                                     included
   distress language (NOT AVAILABLE -- no text corpus) excluded -- Phase 2b data acquisition needed

[done] wrote ../reports/experiment5_fusion_partial.csv
[done] median RHI across 1892 scored companies: 56.4
